# 03 · Classification: metrics, imbalance and honest evaluation

Chapter 3 of the book opens with a warning: **accuracy is generally not the preferred
performance measure for classifiers**, especially on skewed datasets. This tutorial
shows how the `scoring` argument changes what gets tuned, how to evaluate the final
model on the test split with the full confusion matrix, and how stratification and
label types behave.

In [1]:
import warnings

warnings.filterwarnings("ignore")  # hide convergence and progress-bar warnings in tutorial output

import optuna

optuna.logging.set_verbosity(optuna.logging.ERROR)  # Optuna otherwise logs every trial, and a traceback per failed one

## 1. An imbalanced problem

5 % positives. A classifier that always predicts the majority class scores 95 % accuracy
and is useless. `DummyClassifier` is a scikit-learn classifier like any other, so the
same API produces the baseline.

In [2]:
from sklearn.datasets import make_classification

from mloptune import FineTuneConfig, FineTuner

X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=6, n_redundant=4,
    weights=[0.95, 0.05], flip_y=0.01, random_state=42,
)
print("positive rate:", y.mean())

baseline = FineTuner(
    FineTuneConfig("DummyClassifier", "classification", "tutorial-03-imbalanced",
                   model_kwargs={"strategy": "most_frequent"})
).run(X, y)
print("majority-class baseline accuracy:", round(baseline.test_score, 4))

positive rate: 0.054


majority-class baseline accuracy: 0.945


## 2. The metric you tune on is the metric you get

Same model, same search space, same split (same `experiment_name`), three different
`scoring` values. Any name from `sklearn.metrics.get_scorer_names()` works, and so does a
`make_scorer(...)` callable.

In [3]:
from sklearn.metrics import fbeta_score, get_scorer, make_scorer

search_space = {
    "C": {"type": "float", "low": 1e-3, "high": 1e2, "log": True},
    "class_weight": {"type": "categorical", "choices": [None, "balanced"]},
}
scorings = {
    "accuracy": "accuracy",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "f2 (recall-heavy)": make_scorer(fbeta_score, beta=2),
}

runs = {}
for label, scoring in scorings.items():
    config = FineTuneConfig(
        model_name="LogisticRegression",
        problem_type="classification",
        experiment_name="tutorial-03-imbalanced",
        scoring=scoring,
        n_trials=10,
        search_space=search_space,
        model_kwargs={"max_iter": 2000},
    )
    runs[label] = FineTuner(config).run(X, y)
    print(f"{label:18s} best={runs[label].best_params}  test {label}={runs[label].test_score:.4f}")

accuracy           best={'C': 0.01862932117830701, 'class_weight': None}  test accuracy=0.9525
f1                 best={'C': 3.7138230823797542, 'class_weight': None}  test f1=0.3077
roc_auc            best={'C': 0.2422907119919375, 'class_weight': 'balanced'}  test roc_auc=0.8586
f2 (recall-heavy)  best={'C': 0.005688541671741688, 'class_weight': 'balanced'}  test f2 (recall-heavy)=0.4518


Tuning on accuracy kept `class_weight=None` and landed barely above the dummy baseline.
Plain F1 also kept the default weighting, whereas ROC AUC and the recall-heavy F2 scorer
chose `class_weight="balanced"`. Note that each `test_score` is in its own metric, so the
rows are not comparable to each other, only to their own baselines.

## 3. Evaluate the final model properly on the test split

`FineTuneResult` carries the model and the seed, not the data. To get the exact test rows
back, call `split_dataset` with the same sizes, the result's seed, and `stratify=y`
(the tuner stratifies for classification). Then any scikit-learn metric applies.

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

from mloptune import split_dataset

chosen = runs["f2 (recall-heavy)"]
split = split_dataset(X, y, test_size=0.2, validation_size=0.2, random_state=chosen.seed, stratify=y)
assert len(split["y_test"]) == chosen.split_sizes["test"]

y_pred = chosen.model.predict(split["X_test"])
print(confusion_matrix(split["y_test"], y_pred))
print(classification_report(split["y_test"], y_pred, digits=3))

[[315  63]
 [  7  15]]
              precision    recall  f1-score   support

           0      0.978     0.833     0.900       378
           1      0.192     0.682     0.300        22

    accuracy                          0.825       400
   macro avg      0.585     0.758     0.600       400
weighted avg      0.935     0.825     0.867       400



Compare with the accuracy-tuned model on the very same rows:

In [5]:
y_pred_acc = runs["accuracy"].model.predict(split["X_test"])
print(confusion_matrix(split["y_test"], y_pred_acc))
print(classification_report(split["y_test"], y_pred_acc, digits=3))

[[378   0]
 [ 19   3]]
              precision    recall  f1-score   support

           0      0.952     1.000     0.975       378
           1      1.000     0.136     0.240        22

    accuracy                          0.953       400
   macro avg      0.976     0.568     0.608       400
weighted avg      0.955     0.953     0.935       400



The accuracy-tuned model finds fewer positives (lower recall). Which trade-off is right
depends on the cost of a miss versus a false alarm, which is a business decision the
metric should encode, exactly as chapter 3 argues with its precision/recall trade-off.

## 4. Multiclass metrics

Binary metric names (`f1`, `roc_auc`) fail on more than two classes. Use their averaged
variants: `f1_macro`, `f1_weighted`, `roc_auc_ovr`, `roc_auc_ovo`.

In [6]:
from sklearn.datasets import load_wine

wine = load_wine()
for scoring in ("f1_macro", "roc_auc_ovr"):
    config = FineTuneConfig(
        model_name="RandomForestClassifier",
        problem_type="classification",
        experiment_name="tutorial-03-wine",
        scoring=scoring,
        n_trials=6,
        search_space={"max_depth": {"type": "int", "low": 2, "high": 10},
                      "min_samples_leaf": {"type": "int", "low": 1, "high": 8}},
        model_kwargs={"n_estimators": 100},
    )
    wine_result = FineTuner(config).run(wine.data, wine.target)
    print(f"{scoring:12s} best={wine_result.best_params} test={wine_result.test_score:.4f}")

f1_macro     best={'max_depth': 6, 'min_samples_leaf': 1} test=1.0000


roc_auc_ovr  best={'max_depth': 6, 'min_samples_leaf': 1} test=1.0000


## 5. Stratification keeps class proportions in every split

The book's stratified sampling argument (chapter 2, "Create a Test Set") is applied
automatically: both splits use `stratify=y`, so a rare class is represented
proportionally in train, validation and test.

In [7]:
import numpy as np

for name in ("y_train", "y_validation", "y_test"):
    part = split[name]
    print(f"{name:13s} n={len(part):4d}  positive rate={part.mean():.3f}")
print(f"{'whole dataset':13s} n={len(y):4d}  positive rate={y.mean():.3f}")

y_train       n=1200  positive rate=0.054
y_validation  n= 400  positive rate=0.052
y_test        n= 400  positive rate=0.055
whole dataset n=2000  positive rate=0.054


## 6. String labels

scikit-learn models accept string class labels directly. XGBoost does not; encode them
first with `LabelEncoder` and decode predictions afterwards.

In [8]:
from sklearn.preprocessing import LabelEncoder

from mloptune import framework

labels = np.array(["cultivar-A", "cultivar-B", "cultivar-C"])[wine.target]  # string class labels
sk = FineTuner(FineTuneConfig("DecisionTreeClassifier", "classification", "tutorial-03-strings")).run(wine.data, labels)
print("scikit-learn with string labels -> predicts", sk.model.predict(wine.data[[0, 70, 140]]))

if framework.XGBClassifier is None:
    print("xgboost is not available in this environment (on macOS: brew install libomp); skipping the XGBoost example.")
else:
    encoder = LabelEncoder().fit(labels)
    xgb = FineTuner(
        FineTuneConfig("XGBClassifier", "classification", "tutorial-03-strings", model_kwargs={"n_estimators": 50})
    ).run(wine.data, encoder.transform(labels))
    print("XGBoost with encoded labels    -> predicts", encoder.inverse_transform(xgb.model.predict(wine.data[[0, 70, 140]])))

scikit-learn with string labels -> predicts ['cultivar-A' 'cultivar-B' 'cultivar-C']


XGBoost with encoded labels    -> predicts ['cultivar-A' 'cultivar-B' 'cultivar-C']


## Takeaways

- Pick `scoring` to match the cost of errors; never default to accuracy on skewed data.
- Rebuild the split with `split_dataset(..., random_state=result.seed, stratify=y)` to run any
  metric or plot on the held-out test rows.
- Use averaged metric names for multiclass problems.